[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/josh-nowak/bundeswebsites/blob/main/bundeswebsites.ipynb)

# Setup

In [ ]:
# If using Google Colab, install the required packages by running:
# !pip install beautifulsoup4 camelot-py ghostscript pandas pypdfium2 requests tqdm

In [ ]:
import camelot
import pandas as pd
import re
import requests
from urllib.parse import urlparse, urlunparse
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from bs4 import BeautifulSoup

# Data import

In [ ]:
pdf_url = "https://dserver.bundestag.de/btd/20/150/2015028.pdf"
tables = camelot.read_pdf(pdf_url, pages="79-end", backend ="pdfium") # ~1 min
tables

# Basic cleaning

In [ ]:
# Merge tables
df = pd.concat((table.df for table in tables))
df = df.rename(columns={0: "ressort", 1: "url"})

# Remove "header rows"
df = df[df["ressort"] != "Ressort"]

# Remove newlines
df["ressort"] = df["ressort"].str.replace("\n", "", regex=True)
df["url"] = df["url"].str.replace("\n", "", regex=True)

# Remove "Website " or "Websites " prefix in some rows
df["url"] = df["url"].str.replace("Website(s?) ", "", regex=True)

# Split rows with muliple URLs
df["url"] = df["url"].str.split(" ")
df = df.explode("url")

# Simplify BPA entries
df.loc[df["ressort"].str.contains("BPA"), "ressort"] = "BPA"

# Order alphabetically
df = df.sort_values(by=["ressort", "url"])

# Reset index
df = df.reset_index(drop=True)

df

# URL validation and repairing

In [ ]:
def clean_url(url):
    if pd.isna(url):
            return None

    # Convert to string if not already
    url = str(url).strip()

    # Valid URLs should contain a domain with at least one dot
    domain_pattern = r"([a-zA-Z0-9]([a-zA-Z0-9\-]{0,61}[a-zA-Z0-9])?\.)+[a-zA-Z]{2,}"

    # If it doesn"t have a scheme but matches domain pattern, add https://
    if not re.match(r"^https?://", url, re.IGNORECASE):
        if re.search(domain_pattern, url):
            url = "https://" + url
        else:
            # Not a valid URL
            return None

    # Further validate by parsing
    try:
        parsed = urlparse(url)
        # Check if the netloc (domain) part exists and contains at least one dot
        if not parsed.netloc or "." not in parsed.netloc:
            return None

        # Standardize to https
        if parsed.scheme == "http":
            scheme = "https"
        elif parsed.scheme == "https":
            scheme = "https"
        else:
            return None  # Invalid scheme

        # Rebuild URL with consistent format
        url = urlunparse((
            scheme,
            parsed.netloc.rstrip("/"),
            parsed.path,
            parsed.params,
            parsed.query,
            parsed.fragment
        ))
        return url
    except:
        return None  # Invalid URL

df["url_clean"] = df["url"].apply(clean_url)

df[pd.isna(df["url_clean"])]

Rows that could not be parsed to valid URLs are mostly part of descriptions (e.g., "Mitarbeiterportal", "Virtual Reality Projekt").

Only two rows could not have been parsed to URLs and are corrected manually below.

In [ ]:
df.loc[df["url"] == "aussiedlerbeauftragter,info", "url_clean"] = "aussiedlerbeauftragter.info"
df.loc[df["url"] == "https://https://www.env-it.de/stationen", "url_clean"] = "https://www.env-it.de/stationen"

In [ ]:
df = df.drop(columns=["url"])
df = df.rename(columns={"url_clean": "url"})
df = df.dropna()
df

In [ ]:
# Remove duplicates
df = df.drop_duplicates(keep="first")
df = df.reset_index(drop=True)
df

Some URLs appear with multiple organisations (e.g., deutschlandatlas.bund.de). Such duplicates are not removed.

# Adding response status code

In [ ]:
def get_status_code(url):
    try:
        response = requests.get(url, timeout=10, allow_redirects=True)
        
        # Get initial status (first response in history, or the response itself if no redirects)
        initial_status = str(response.history[0].status_code if response.history else response.status_code)
        
        # Get final status
        final_status = str(response.status_code)
    except requests.exceptions.RequestException:
        initial_status = "unavailable"
        final_status = "unavailable"
        
    return url, initial_status, final_status

def fetch_status_codes_parallel(urls, max_workers=10):
    
    initial_status_dict = {}
    final_status_dict = {}
    unique_urls = list(set(urls))

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_url = {executor.submit(get_status_code, url): url for url in unique_urls}
        for future in tqdm(as_completed(future_to_url), total=len(urls), desc="Fetching status codes"):
            try:
                url, initial_status, final_status = future.result()
                initial_status_dict[url] = initial_status
                final_status_dict[url] = final_status
            except Exception as e:
                print(f"Error processing URL: {e}")
                initial_status_dict[url] = "error"
                final_status_dict[url] = "error"
    
    return initial_status_dict, final_status_dict


urls = df["url"].tolist()
initial_dict, final_dict = fetch_status_codes_parallel(urls)
df_with_status = df.copy()
df_with_status["initial_status"] = df_with_status["url"].map(initial_dict)
df_with_status["final_status"] = df_with_status["url"].map(final_dict)
df_with_status

In [ ]:
# df_with_status.to_csv("df_with_status.csv", index=False)
# df_with_status = pd.read_csv("df_with_status.csv")

In [ ]:
df_with_status.value_counts("final_status")

# Add page info

In [ ]:
def get_page_info(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept-Language': 'de-DE,de;q=0.9,en-US;q=0.8,en;q=0.7', # Prioritize German (Germany), then any German, then English
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8',
    }

    result = {
        "title": None,
        "description": None,
        "hreflang": None,
        "canonical_url": None,
        "uses_gsb": False,
    }
    
    try:           
        response = requests.get(url, headers=headers, timeout=15, allow_redirects=True)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Get title
        result["title"] = soup.title.string if soup.title and soup.title.string else None
        
        # Get meta description
        meta_desc_tag = soup.find("meta", attrs={"name": "description"}) or soup.find("meta", attrs={"property": "og:description"})
        if meta_desc_tag and meta_desc_tag.get("content"):
            result["description"] = meta_desc_tag.get("content")

        # Get hreflang links
        # TODO: Consider only storing languages instead of full URLs
        hreflang_tags = soup.find_all("link", rel="alternate", href=True, hreflang=True)
        if hreflang_tags:
            result["hreflang"] = ", ".join(tag.get("hreflang").lower() for tag in hreflang_tags)

        # Get canonical link
        canonical_tag = soup.find("link", rel="canonical", href=True)
        if canonical_tag:
            result["canonical_url"] = canonical_tag.get("href")

        # Check if Government Site Builder (GSB) is used
        meta_generator_tag = soup.find("meta", attrs={"name": "generator"})
        if meta_generator_tag and meta_generator_tag.get("content") and \
            "Government Site Builder" in meta_generator_tag.get("content"):
                result["uses_gsb"] = True
        
    except Exception as e:
        print(f"Error fetching page info for {url}: {e}")
    
    return result
    


def fetch_page_info_parallel(urls, max_workers=10):
    results_dict = {}
    unique_urls = list(set(urls))

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_url = {executor.submit(get_page_info, url): url for url in unique_urls}
        for future in tqdm(as_completed(future_to_url), total=len(urls), desc="Fetching page info"):
            url = future_to_url[future]
            try:
                page_data = future.result()
                results_dict[url] = page_data
            except Exception as e:
                print(f"Error processing URL {url}: {e}")
                
    return results_dict

reachable_urls = df_with_status[df_with_status["final_status"] == "200"]["url"].tolist()
page_data = fetch_page_info_parallel(reachable_urls)

In [ ]:
page_data_df = pd.DataFrame.from_dict(page_data, orient="index")
page_data_df = page_data_df.reset_index()
page_data_df = page_data_df.rename(columns={"index": "url"})

df_final = pd.merge(df_with_status, page_data_df, on="url", how="left")
df_final

# Export

In [ ]:
df_final.to_csv("bundeswebsites.csv", index=False)